#Install Needed Dependencies

In [ ]:
!pip install pytorch-metric-learning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.9/125.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 62.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [ ]:
!pip install torch torchvision pyyaml scikit-learn

In [ ]:
from torchvision import datasets

# ACMMM23-Solution-MBEG

In [ ]:
!git clone https://github.com/Reza-Zhu/ACMMM23-Solution-MBEG.git /content/ACMMM23-Solution-MBEG

%cd /content/ACMMM23-Solution-MBEG

Cloning into '/content/ACMMM23-Solution-MBEG'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 45 (delta 17), reused 20 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 8.80 MiB | 5.12 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/ACMMM23-Solution-MBEG


In [ ]:
import os
weights_path = "/content/ACMMM23-Solution-MBEG/weights"

os.makedirs(weights_path, exist_ok=True)

print(f"Directory created at: {weights_path}")

Directory created at: /content/ACMMM23-Solution-MBEG/weights


In [ ]:
import yaml

with open("/content/ACMMM23-Solution-MBEG/settings.yaml", 'r') as f:
    config = yaml.safe_load(f)

    config['dataset_path'] = '/content/dataset_folder/University-Release'
    config['weight_save_path'] = weights_path
    config['num_epochs'] = 10
    config['batch_size'] = 4
    config['model'] = 'ResNet'
    config['name'] = 'ResNet_1652'

    with open("settings.yaml", 'w') as f:
        yaml.dump(config, f)

In [ ]:
import os
os.chdir("/content/ACMMM23-Solution-MBEG")

# Modifying Data Preprocessing & Model Definition

In [ ]:
code = '''
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from pathlib import Path


class DualResNet(nn.Module):
    def __init__(self, num_classes=1652, pretrained=True, dropout=0.3):
        super(DualResNet, self).__init__()
        self.backbone1 = models.resnet18(pretrained=pretrained)
        self.backbone2 = models.resnet18(pretrained=pretrained)
        self.backbone1.fc = nn.Identity()
        self.backbone2.fc = nn.Identity()
        self.classifier1 = nn.Sequential(nn.Dropout(dropout), nn.Linear(512, num_classes))
        self.classifier2 = nn.Sequential(nn.Dropout(dropout), nn.Linear(512, num_classes))

    def forward(self, x1, x2):
        f1 = self.backbone1(x1)
        f2 = self.backbone2(x2)
        out1 = self.classifier1(f1)
        out2 = self.classifier2(f2)
        return out1, out2, f1, f2

def get_num_classes(data_dir):
      sat_dir = Path(data_dir) / "satellite"
      class_folders = [f for f in sat_dir.iterdir() if f.is_dir()]
      return len(class_folders)
'''
with open("modified_dualresnet.py", "w") as f:
        f.write(code)

In [ ]:
code = '''
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from pathlib import Path

class PairedU1652Dataset(Dataset):
    def __init__(self, root_dir, transform_sat=None, transform_drone=None):
        self.root_dir = root_dir

        self.sat_dir = os.path.join(root_dir, 'satellite')
        self.drone_dir = os.path.join(root_dir, 'drone')

        self.transform_sat = transform_sat
        self.transform_drone = transform_drone

        self.sat_classes = sorted(os.listdir(self.sat_dir))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.sat_classes)}

        self.sat_images = []
        self.drone_images = []
        self.labels = []

        for cls_name in self.sat_classes:
            sat_cls_path = os.path.join(self.sat_dir, cls_name)
            drone_cls_path = os.path.join(self.drone_dir, cls_name)

            sat_imgs = sorted(os.listdir(sat_cls_path))
            drone_imgs = sorted(os.listdir(drone_cls_path))

            assert len(sat_imgs) == 1, f"Expected exactly 1 satellite image per class '{cls_name}', but got {len(sat_imgs)}"

            sat_img_name = sat_imgs[0]
            sat_img_path = os.path.join(sat_cls_path, sat_img_name)

            for drone_img_name in drone_imgs:
                drone_img_path = os.path.join(drone_cls_path, drone_img_name)

                self.sat_images.append(sat_img_path)       # Repeat the same satellite image path
                self.drone_images.append(drone_img_path)
                self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        sat_img_path = self.sat_images[idx]
        drone_img_path = self.drone_images[idx]
        label = self.labels[idx]

        sat_img = Image.open(sat_img_path).convert('RGB')
        drone_img = Image.open(drone_img_path).convert('RGB')

        if self.transform_sat:
            sat_img = self.transform_sat(sat_img)
        if self.transform_drone:
            drone_img = self.transform_drone(drone_img)

        return sat_img, drone_img, label

'''
with open("Modified_Preprocessing.py", "w") as f:
    f.write(code)


In [ ]:
import importlib
import Modified_Preprocessing
importlib.reload(Modified_Preprocessing)


<module 'Modified_Preprocessing' from '/content/ACMMM23-Solution-MBEG/Modified_Preprocessing.py'>

# Test Codalab

In [ ]:
!pip install --quiet gdown  # install gdown if not already installed

import gdown

file_id = '1pBWbaZPB-djPrVnqe8fuzoFYRuYk9mTo'
output = 'gallery_satellite_160k.tar.gz'
gdown.download(f'https://drive.google.com/uc?id={file_id}', output, quiet=False)



Downloading...
From (original): https://drive.google.com/uc?id=1pBWbaZPB-djPrVnqe8fuzoFYRuYk9mTo
From (redirected): https://drive.google.com/uc?id=1pBWbaZPB-djPrVnqe8fuzoFYRuYk9mTo&confirm=t&uuid=ae13a333-9cc9-4a32-a59a-ead80524a2c9
To: /content/ACMMM23-Solution-MBEG/gallery_satellite_160k.tar.gz
100%|██████████| 6.09G/6.09G [01:33<00:00, 65.4MB/s]


'gallery_satellite_160k.tar.gz'

In [ ]:
import os
print(f"File size in Colab: {os.path.getsize('gallery_satellite_160k.tar.gz') / (1024*1024):.2f} MB")


File size in Colab: 5812.22 MB


In [ ]:
mkdir -p /content/gallery_satellite_160k/


In [ ]:
!apt install pigz
!unpigz -c gallery_satellite_160k.tar.gz | tar xvf - -C /content/gallery_satellite_160k/

Streaming output truncated to the last 5000 lines.
gallery_satellite_160k/UJLq8VzTPgDeXpC.webp
gallery_satellite_160k/6a4wQRoZ1k8V5Jm.webp
gallery_satellite_160k/N7riOajBBkBODJl.webp
gallery_satellite_160k/vvz7iT5mUJHDshQ.webp
gallery_satellite_160k/JBZGsPjaj3TRSWF.webp
gallery_satellite_160k/ByBOBnDSx5lDeZq.webp
gallery_satellite_160k/KFPJ9a8MRfWdWZa.webp
gallery_satellite_160k/fNQpDIcKID4kD0C.webp
gallery_satellite_160k/6bW2j1hAtci7dFz.webp
gallery_satellite_160k/36sNG46lohS8iTY.webp
gallery_satellite_160k/h46t0m32iBjSl2J.webp
gallery_satellite_160k/leZjGXpRIkvt5ow.webp
gallery_satellite_160k/XvhWodFaqp7Pb2e.webp
gallery_satellite_160k/NZ40dzp9YdpeQzW.webp
gallery_satellite_160k/8mNE6CdWqhbRTAc.webp
gallery_satellite_160k/FA4NUlJhpZk2vsK.webp
gallery_satellite_160k/caesQgeHOEjF1Yb.webp
gallery_satellite_160k/zLsoP4HjvTKUoKc.webp
gallery_satellite_160k/4ReL7AOxb4uwkBK.webp
gallery_satellite_160k/9ZyggaoUxQYGZxY.webp
gallery_satellite_160k/KOdgrhG6gV8Q8A3.webp
gallery_satellite_160k/IR

# Test


In [ ]:
!pip install --quiet gdown

import gdown

file_id = '1w50Qqff1YwQ-c3xMgLOelgYuHDnAbDz7'
output = 'query_drone160k_wx.zip'  # <-- include .zip

gdown.download(f'https://drive.google.com/uc?id={file_id}', output, quiet=False)


Downloading...
From (original): https://drive.google.com/uc?id=1w50Qqff1YwQ-c3xMgLOelgYuHDnAbDz7
From (redirected): https://drive.google.com/uc?id=1w50Qqff1YwQ-c3xMgLOelgYuHDnAbDz7&confirm=t&uuid=229d8128-623d-416e-99bb-cdc5f6a98016
To: /content/ACMMM23-Solution-MBEG/query_drone160k_wx.zip
100%|██████████| 2.35G/2.35G [00:34<00:00, 68.6MB/s]


'query_drone160k_wx.zip'

In [ ]:
!unzip query_drone160k_wx.zip -d /content/query_drone160k_wx/


Streaming output truncated to the last 5000 lines.
  inflating: /content/query_drone160k_wx/query_drone_160k_wx_24/18Joq19JYbtdn4z.jpeg  
  inflating: /content/query_drone160k_wx/query_drone_160k_wx_24/xCIM9Upq5WwxKXy.jpeg  
  inflating: /content/query_drone160k_wx/query_drone_160k_wx_24/wqxHk79HpvB4c3Z.jpeg  
  inflating: /content/query_drone160k_wx/query_drone_160k_wx_24/74PdBBajAj4lRRo.jpeg  
  inflating: /content/query_drone160k_wx/query_drone_160k_wx_24/BhK5F8R1Gar5tFj.jpeg  
  inflating: /content/query_drone160k_wx/query_drone_160k_wx_24/HNmLv18JLQC0vcU.jpeg  
  inflating: /content/query_drone160k_wx/query_drone_160k_wx_24/dqFfEz7w1o9QEbk.jpeg  
  inflating: /content/query_drone160k_wx/query_drone_160k_wx_24/QbS4Lwx7Bj7Fm4r.jpeg  
  inflating: /content/query_drone160k_wx/query_drone_160k_wx_24/EEDDVSxbklY9Zy5.jpeg  
  inflating: /content/query_drone160k_wx/query_drone_160k_wx_24/rwRsT9MBp6lFt3n.jpeg  
  inflating: /content/query_drone160k_wx/query_drone_160k_wx_24/kGAZNEZ71hlaBqO

In [ ]:
import os

file_path = "/content/gallery_satellite_160k/gallery_satellite_160k/3crDtFh6au3itPt.webp"

if os.path.exists(file_path):
    os.remove(file_path)
    print("File removed.")
else:
    print("File not found.")


File removed.


In [ ]:
import os
import numpy as np
import torch
from torchvision import transforms
from PIL import Image
import cv2
from sklearn.metrics.pairwise import cosine_similarity
from modified_dualresnet import DualResNet  # <-- Ensure this is your custom model file

# --- Paths ---
gallery_dir = '/content/gallery_satellite_160k/gallery_satellite_160k'
query_dir = '/content/query_drone160k_wx/query_drone_160k_wx_24'
query_list_file = '/content/query_drone_name.txt'
weights_path = '/content/ACMMM23-Solution-MBEG/weights/finetuned_best.pth'

# --- Load query filenames ---
with open(query_list_file, 'r') as f:
    query_filenames = [line.strip() for line in f.readlines()]

# --- Load gallery filenames ---
valid_exts = ('.jpg', '.jpeg', '.png', '.webp')
gallery_filenames = [
    fname for fname in sorted(os.listdir(gallery_dir))
    if fname.lower().endswith(valid_exts)
]

print(f"Gallery images found: {len(gallery_filenames)}")
print(f"Query images found: {len(query_filenames)}")

# --- Device setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Load DualResNet ---
model = DualResNet(num_classes=100, dropout=0.3)
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()

# --- Image preprocessing ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# --- Feature extractor using DualResNet (fusion from both branches) ---
def extract_dualresnet_feature(image_path, view=1):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"OpenCV could not read image: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img)
    img_tensor = transform(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        if view == 1:
            feat1, _, _, _ = model(img_tensor, torch.zeros_like(img_tensor))
            return feat1.squeeze().cpu().numpy()
        else:
            _, feat2, _, _ = model(torch.zeros_like(img_tensor), img_tensor)
            return feat2.squeeze().cpu().numpy()

# --- Extract features with fusion (concat of both views) ---
def fused_feature(image_path):
    f1 = extract_dualresnet_feature(image_path, view=1)
    f2 = extract_dualresnet_feature(image_path, view=2)
    fused = np.concatenate([f1, f2])
    norm = np.linalg.norm(fused)
    return fused / norm if norm > 0 else fused

# --- Extract gallery features ---
gallery_features = []
valid_gallery_filenames = []

print("Extracting gallery features...")
for fname in gallery_filenames:
    try:
        feat = fused_feature(os.path.join(gallery_dir, fname))
        gallery_features.append(feat)
        valid_gallery_filenames.append(fname)
    except Exception as e:
        print(f"Failed gallery image {fname}: {e}")

gallery_features = np.vstack(gallery_features)
print(f"Extracted {len(gallery_features)} gallery features.")

# --- Extract query features ---
query_features = []
valid_query_filenames = []

print("Extracting query features...")
for fname in query_filenames:
    try:
        feat = fused_feature(os.path.join(query_dir, fname))
        query_features.append(feat)
        valid_query_filenames.append(fname)
    except Exception as e:
        print(f"Failed query image {fname}: {e}")

query_features = np.vstack(query_features)
print(f"Extracted {len(query_features)} query features.")

# --- Compute cosine similarity ---
print("Computing cosine similarity...")
sim_matrix = cosine_similarity(query_features, gallery_features)

# --- Write answer.txt ---
output_file = 'answer.txt'
print(f"Writing to {output_file} ...")

with open(output_file, 'w') as f:
    for i, query_name in enumerate(valid_query_filenames):
        sim_scores = sim_matrix[i]
        top_indices = np.argsort(sim_scores)[::-1][:10]

        query_base = os.path.splitext(query_name)[0]
        top_gallery_bases = [os.path.splitext(valid_gallery_filenames[idx])[0] for idx in top_indices]

        line = query_base + ' ' + ' '.join(top_gallery_bases) + '\n'
        f.write(line)

print("Done! Output saved in answer.txt.")


FileNotFoundError: [Errno 2] No such file or directory: '/content/query_drone_name.txt'